# Connect IDE Clients to MCP Servers on OpenShift

Once your MCP servers are deployed, configure your IDE to connect via the OpenShift Route URLs.

## 1. Get Your Route URLs

In [ ]:
%%bash
echo "=== MCP Server Endpoints ==="
echo ""
echo "Context7 (remote):          https://mcp.context7.com/mcp"
echo ""
for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    printf "%-30s https://%s/sse\n" "${route}:" "${host}"
done

Example output:
```
Context7 (remote):              https://mcp.context7.com/mcp

mcp-sequential-thinking:        https://mcp-sequential-thinking-mcp-servers.apps.cluster.example.com/sse
mcp-github:                     https://mcp-github-mcp-servers.apps.cluster.example.com/sse
mcp-gh-grep:                    https://mcp-gh-grep-mcp-servers.apps.cluster.example.com/sse
mcp-chrome-devtools:            https://mcp-chrome-devtools-mcp-servers.apps.cluster.example.com/sse
```

## 2. Generate IDE Config Files

Run the cell below to auto-generate MCP config files for each IDE using your actual cluster domain.

In [ ]:
import subprocess
import json

result = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()

# Get deployed routes
routes_result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

mcp_urls = {"context7": "https://mcp.context7.com/mcp"}
for line in routes_result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        short_name = name.replace("mcp-", "")
        mcp_urls[short_name] = f"https://{host}/sse"

print(f"Cluster domain: {CLUSTER_DOMAIN}")
print(f"Discovered {len(mcp_urls)} MCP servers")
for name, url in mcp_urls.items():
    print(f"  {name}: {url}")

## 3. VS Code (Agent Mode)

Create `.vscode/mcp.json` in your project root:

In [ ]:
vscode_config = {
    "servers": {}
}

type_map = {"context7": "http"}

for name, url in mcp_urls.items():
    vscode_config["servers"][name] = {
        "type": type_map.get(name, "sse"),
        "url": url
    }

print("=== .vscode/mcp.json ===")
print(json.dumps(vscode_config, indent=2))
print("\nCommit this file to your repo — all team members get the same MCP tools automatically.")

## 4. Cursor

Create `.cursor/mcp.json` in your project root:

In [ ]:
cursor_config = {
    "mcpServers": {}
}

for name, url in mcp_urls.items():
    cursor_config["mcpServers"][name] = {"url": url}

print("=== .cursor/mcp.json ===")
print(json.dumps(cursor_config, indent=2))

## 5. Claude Code

Create `.mcp.json` in project root (or `~/.claude/mcp.json` for global):

In [ ]:
claude_config = {
    "mcpServers": {}
}

for name, url in mcp_urls.items():
    claude_config["mcpServers"][name] = {"type": "url", "url": url}

print("=== .mcp.json (Claude Code) ===")
print(json.dumps(claude_config, indent=2))

## 6. OpenCode

Edit `~/.config/opencode/config.json`:

In [ ]:
opencode_config = {
    "mcp": {}
}

for name, url in mcp_urls.items():
    opencode_config["mcp"][name] = {"type": "remote", "url": url}

print("=== ~/.config/opencode/config.json ===")
print(json.dumps(opencode_config, indent=2))

## 7. Team Deployment Tips

### Shared Project Config

Commit the MCP config to your project repository so all team members get the same tools:

```bash
# For a team using VS Code
git add .vscode/mcp.json
git commit -m "Add shared MCP server configuration"
```

### DNS Alias (Optional)

For cleaner URLs, create a CNAME or use OpenShift Route with a custom hostname:

```yaml
apiVersion: route.openshift.io/v1
kind: Route
metadata:
  name: mcp-github-custom
  namespace: mcp-servers
spec:
  host: mcp-github.company.com
  to:
    kind: Service
    name: mcp-github
  tls:
    termination: edge
```

### Authentication (Optional)

For production, add OAuth proxy sidecar to MCP server pods:

```yaml
containers:
  - name: oauth-proxy
    image: registry.redhat.io/openshift4/ose-oauth-proxy:v4.14
    args:
      - --upstream=http://localhost:3002
      - --cookie-secret=SECRET
      - --openshift-service-account=mcp-github
```

## 8. Verification

After configuring your IDE:

In [ ]:
import subprocess

print("MCP Server Health Check")
print("=" * 60)

# Check Context7
r = subprocess.run(["curl", "-sk", "-o", "/dev/null", "-w", "%{http_code}", "-m", "5",
                    "https://mcp.context7.com/mcp"], capture_output=True, text=True)
status = "✅" if r.stdout.strip() in ["200", "405"] else "❌"
print(f"{status} Context7 (remote): https://mcp.context7.com/mcp")

# Check deployed servers
result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/sse"
        r = subprocess.run(["curl", "-sk", "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
                          capture_output=True, text=True)
        status = "✅" if r.stdout.strip() in ["200", "405"] else "❌"
        print(f"{status} {name}: {url}")

print("\nIDE-specific verification:")
print("  VS Code:    Command Palette → 'MCP: List Servers' → check green status")
print("  Cursor:     Settings → MCP → verify connected indicators")
print("  Claude Code: Run /mcp to list connected servers")
print("  OpenCode:   Type /mcp to see available tools")